In [ ]:
!pip -q install imagehash open_clip_torch pillow pandas

In [ ]:
import glob, itertools, os
import imagehash, numpy as np, pandas as pd, torch, open_clip
from PIL import Image, ImageOps

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai")
model = model.to(device).eval()

def buka(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")

def skor_ahash(a, b):
    return 1 - (imagehash.average_hash(a) - imagehash.average_hash(b)) / 64

def skor_phash(a, b):
    return 1 - (imagehash.phash(a) - imagehash.phash(b)) / 64

@torch.no_grad()
def embed(img):
    x = preprocess(img).unsqueeze(0).to(device)
    v = model.encode_image(x)
    return (v / v.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

def skor_clip(a, b):
    return float(np.dot(embed(a), embed(b)))

In [ ]:
ids = sorted({os.path.basename(p)[:2] for p in glob.glob("foto-uji/*_a.jpeg")})
A = {i: buka(f"foto-uji/{i}_a.jpeg") for i in ids}
B = {i: buka(f"foto-uji/{i}_b.jpeg") for i in ids}

rows = []
for i in ids:                                  # cocok: benda sama
    rows.append((i, i, 1, A[i], B[i]))
for i, j in itertools.permutations(ids, 2):    # tidak cocok: benda beda
    rows.append((i, j, 0, A[i], B[j]))

data = []
for i, j, label, a, b in rows:
    data.append(dict(a=i, b=j, label=label,
                     ahash=skor_ahash(a, b), phash=skor_phash(a, b),
                     clip=skor_clip(a, b)))
df = pd.DataFrame(data)
print(len(df), "pasangan:", df.label.sum(), "cocok,", (df.label == 0).sum(), "tidak cocok")

In [ ]:
def ringkas(metode):
    cocok = df[df.label == 1][metode]
    beda = df[df.label == 0][metode]
    # cari threshold dengan akurasi terbaik
    best_t, best_acc = 0, 0
    for t in np.linspace(df[metode].min(), df[metode].max(), 200):
        acc = ((df[metode] >= t) == (df.label == 1)).mean()
        if acc > best_acc: best_t, best_acc = t, acc
    # AUC sederhana: peluang skor cocok > skor tidak cocok
    auc = np.mean([c > b for c in cocok for b in beda])
    return dict(metode=metode, rata_cocok=round(cocok.mean(), 3),
                rata_beda=round(beda.mean(), 3),
                selisih=round(cocok.mean() - beda.mean(), 3),
                AUC=round(auc, 3), threshold=round(best_t, 3),
                akurasi=round(best_acc, 3))

print(pd.DataFrame([ringkas(m) for m in ["ahash", "phash", "clip"]]).to_string(index=False))

In [ ]:
dompet_ids = ["13","14","15","16","17","18","19"]
sub = df[df.a.isin(dompet_ids) & df.b.isin(dompet_ids)]
print(len(sub), "pasangan:", sub.label.sum(), "cocok,", (sub.label == 0).sum(), "tidak cocok")
for m in ["ahash", "phash", "clip"]:
    c = sub[sub.label == 1][m]; b = sub[sub.label == 0][m]
    auc = np.mean([x > y for x in c for y in b])
    print(m, "AUC dompet-vs-dompet:", round(auc, 3),
          "| rata cocok", round(c.mean(), 3), "| rata beda", round(b.mean(), 3))